# ThermoRG Phase B — Session 1: Full Training Validation

## Goal
Train 6 ThermoNet architectures to convergence (200 epochs) on CIFAR-10,
measure final test accuracy, and verify J_topo → E_floor correlation.

## Data Split
- **Training set**: 45,000 images (90% of CIFAR-10 train)
- **Validation set**: 5,000 images (10% of CIFAR-10 train) — used during training for evaluation
- **Test set**: 10,000 images (CIFAR-10 test) — **reserved for final evaluation ONLY**

## Candidates

| ID | J_topo | Params | Selection Reason |
|----|--------|--------|-------------------|
| T18 | 0.789 | 0.59M | Best J_topo |
| T04 | 0.763 | 0.93M | No-skip control |
| T11 | 0.505 | 1.05M | Threshold boundary |
| T09 | 0.749 | 1.31M | High J_topo, mid params |
| T13 | 0.678 | 1.08M | Good J_topo, small params |
| T16 | 0.524 | 2.05M | Threshold boundary |

## Reference Baseline
ResNet-18 (11.7M) achieves ~94.8% on CIFAR-10 (200 epochs, literature value).
**We do NOT retrain ResNet-18 — we use the literature value as benchmark.**

## Success Criteria
ThermoNet with good J_topo (high) should achieve comparable accuracy to ResNet-18
while using significantly fewer parameters.

In [ ]:
import os, sys, time, json, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset, random_split
from torchvision import datasets, transforms
from datetime import datetime

import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
print(f"Start time: {datetime.now().strftime('%H:%M:%S')}")

In [ ]:
# ============================================================
# ThermoNet Model Builders
# ============================================================

def scale_channels(chs, mult):
    return [chs[0]] + [max(1, int(c * mult)) for c in chs[1:]]

class ConvBlock(nn.Module):
    def __init__(self, ic, oc, act='gelu', norm=True):
        super().__init__()
        self.conv = nn.Conv2d(ic, oc, 3, padding=1, bias=not norm)
        self.norm = nn.LayerNorm([oc, 32, 32]) if norm else nn.Identity()
        if act == 'gelu':
            self.act = nn.GELU()
        elif act == 'tga':
            self.act = nn.Tanh()
        else:
            self.act = nn.ReLU(inplace=True)
    
    def forward(self, x):
        return self.act(self.norm(self.conv(x)))

def build_TN3(wm=1.0, num_classes=10, use_skip=True):
    ch = scale_channels([3, 64, 64, 128, 128], wm)
    blocks = nn.ModuleList()
    for i in range(len(ch) - 1):
        blocks.append(ConvBlock(ch[i], ch[i+1], 'gelu', True))
    pool = nn.AdaptiveAvgPool2d((1, 1))
    fc = nn.Linear(ch[-1], num_classes)
    return nn.Sequential(*[*blocks, pool, nn.Flatten(), fc])

def build_TN5(wm=1.0, num_classes=10, use_skip=True):
    ch = scale_channels([3, 64, 128, 256, 128, 64], wm)
    blocks = nn.ModuleList()
    for i in range(len(ch) - 1):
        blocks.append(ConvBlock(ch[i], ch[i+1], 'gelu', True))
    pool = nn.AdaptiveAvgPool2d((1, 1))
    fc = nn.Linear(ch[-1], num_classes)
    return nn.Sequential(*[*blocks, pool, nn.Flatten(), fc])

def build_TN7(wm=1.0, num_classes=10, use_skip=True):
    ch = scale_channels([3, 64, 64, 128, 128, 256, 128, 64], wm)
    blocks = nn.ModuleList()
    for i in range(len(ch) - 1):
        blocks.append(ConvBlock(ch[i], ch[i+1], 'tga', True))
    pool = nn.AdaptiveAvgPool2d((1, 1))
    fc = nn.Linear(ch[-1], num_classes)
    return nn.Sequential(*[*blocks, pool, nn.Flatten(), fc])

def build_TN9(wm=1.0, num_classes=10, use_skip=False):
    ch = scale_channels([3] + [64]*8, wm)
    blocks = nn.ModuleList()
    for i in range(len(ch) - 1):
        blocks.append(ConvBlock(ch[i], ch[i+1], 'gelu', True))
    pool = nn.AdaptiveAvgPool2d((1, 1))
    fc = nn.Linear(ch[-1], num_classes)
    return nn.Sequential(*[*blocks, pool, nn.Flatten(), fc])

def build_TN_arbitrary_depth(depth, wm=1.0, num_classes=10, use_skip=True):
    if depth == 3:
        return build_TN3(wm, num_classes, use_skip)
    elif depth == 5:
        return build_TN5(wm, num_classes, use_skip)
    elif depth == 7:
        return build_TN7(wm, num_classes, use_skip)
    elif depth == 9:
        return build_TN9(wm, num_classes, use_skip)
    else:
        base_pattern = [64, 128, 256, 512]
        ch = [3]
        reps = (depth + 3) // 4
        for _ in range(reps):
            ch.extend(base_pattern)
        ch = ch[:depth+1]
        while len(ch) < depth + 1:
            ch.append(64)
        ch = scale_channels(ch, wm)
        blocks = nn.ModuleList()
        for i in range(len(ch) - 1):
            blocks.append(ConvBlock(ch[i], ch[i+1], 'gelu', True))
        pool = nn.AdaptiveAvgPool2d((1, 1))
        fc = nn.Linear(ch[-1], num_classes)
        return nn.Sequential(*[*blocks, pool, nn.Flatten(), fc])

In [ ]:
# ============================================================
# Data Loading — STRICT train/val/test SPLIT
# ============================================================

# Transforms
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.4914, 0.4822, 0.4465], [0.2023, 0.1994, 0.2010])
])
transform_val = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.4914, 0.4822, 0.4465], [0.2023, 0.1994, 0.2010])
])

# Full CIFAR-10 training set (50,000 images)
full_train_set = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)

# Split into train (45,000) and val (5,000)
n_train = 45000
n_val = 5000
torch.manual_seed(42)
train_subset, val_subset = random_split(
    full_train_set, 
    [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)
# Override val transform (to remove augmentation)
val_set_clean = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_val)
val_indices = val_subset.indices
val_subset = Subset(val_set_clean, val_indices)

# Test set (10,000 images) — NEVER used during training
test_set = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_val)

# DataLoaders
train_loader = DataLoader(train_subset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_subset, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_subset)} images")
print(f"Val:   {len(val_subset)} images")
print(f"Test:  {len(test_set)} images (FINAL EVALUATION ONLY)")
print()
print("NOTE: Val set is used during training for model selection.")
print("      Test set is ONLY used for final evaluation after training completes.")

In [ ]:
# ============================================================
# Training Functions
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion):
    """Train for one epoch. Returns average loss and accuracy."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += X.size(0)
    return total_loss / total, correct / total

def evaluate_on_loader(model, loader):
    """Evaluate model on any loader. Returns average loss and accuracy."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            out = model(X)
            loss = F.cross_entropy(out, y)
            total_loss += loss.item() * X.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += X.size(0)
    return total_loss / total, correct / total

def train_to_convergence(model, train_loader, val_loader, epochs=200, lr=0.1, seed=42):
    """
    Train model to convergence using train_loader.
    Use val_loader for model selection (best checkpoint).
    
    NOTE: test_loader is NOT used during training.
    """
    torch.manual_seed(seed)
    
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4, nesterov=True)
    
    # Cosine annealing scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    
    best_val_acc = 0.0
    best_val_loss = float('inf')
    best_epoch = 0
    best_model_state = None
    
    # Track history on val set
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }
    
    for epoch in range(epochs):
        t0 = time.time()
        
        # Train on training set
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        
        # Evaluate on validation set (NOT test set!)
        val_loss, val_acc = evaluate_on_loader(model, val_loader)
        
        scheduler.step()
        elapsed = time.time() - t0
        
        # Record history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        # Track best model on validation set
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_loss = val_loss
            best_epoch = epoch + 1
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        
        if (epoch + 1) % 20 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:3d}/{epochs}: "
                  f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
                  f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f} ({elapsed:.1f}s)")
    
    # Restore best model
    model.load_state_dict(best_model_state)
    model.to(DEVICE)
    
    return {
        'best_val_acc': best_val_acc,
        'best_val_loss': best_val_loss,
        'best_epoch': best_epoch,
        'history': history
    }

In [ ]:
# ============================================================
# 6 Candidates
# ============================================================

CANDIDATES = [
    {'id': 'T18', 'depth': 9,  'width_mult': 0.5,  'use_skip': True,  'J_topo': 0.789},
    {'id': 'T04', 'depth': 9,  'width_mult': 0.75, 'use_skip': False, 'J_topo': 0.763},
    {'id': 'T11', 'depth': 3,  'width_mult': 1.0,  'use_skip': True,  'J_topo': 0.505},
    {'id': 'T09', 'depth': 9,  'width_mult': 1.0,  'use_skip': True,  'J_topo': 0.749},
    {'id': 'T13', 'depth': 7,  'width_mult': 0.5,  'use_skip': True,  'J_topo': 0.678},
    {'id': 'T16', 'depth': 5,  'width_mult': 1.0,  'use_skip': True,  'J_topo': 0.524},
]

EPOCHS = 200
SEED = 42

# Reference baseline (NOT retrained — literature value)
RESNET18_ACC = 0.948  # He et al. 2016, 200 epochs on CIFAR-10

print(f"Candidates: {[c['id'] for c in CANDIDATES]}")
print(f"Epochs: {EPOCHS}")
print(f"Reference: ResNet-18 accuracy = {RESNET18_ACC:.1%} (literature, NOT retrained)")
print(f"Start time: {datetime.now().strftime('%H:%M:%S')}")

In [ ]:
# ============================================================
# Train All 6 Candidates
# ============================================================

results = []
t_start = time.time()

for i, config in enumerate(CANDIDATES):
    print(f"\n{'='*60}")
    print(f"[{i+1}/{len(CANDIDATES)}] {config['id']}: "
          f"depth={config['depth']}, wm={config['width_mult']}, skip={config['use_skip']}")
    print(f"  Pre-computed J_topo: {config['J_topo']:.4f}")
    
    # Build model
    torch.manual_seed(SEED)
    model = build_TN_arbitrary_depth(
        config['depth'], config['width_mult'], 10, config['use_skip']
    ).to(DEVICE)
    
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Parameters: {n_params/1e6:.2f}M")
    
    # Train using train_loader, select best on val_loader
    print(f"  Training {EPOCHS} epochs...")
    t0 = time.time()
    result = train_to_convergence(model, train_loader, val_loader, epochs=EPOCHS, seed=SEED)
    train_time = time.time() - t0
    
    # FINAL EVALUATION: Use test set ONLY after training is complete
    print(f"  Running final evaluation on test set...")
    test_loss, test_acc = evaluate_on_loader(model, test_loader)
    
    results.append({
        'id': config['id'],
        'depth': config['depth'],
        'width_mult': config['width_mult'],
        'use_skip': config['use_skip'],
        'J_topo': config['J_topo'],
        'n_params': n_params,
        'best_val_acc': result['best_val_acc'],
        'best_val_loss': result['best_val_loss'],
        'best_epoch': result['best_epoch'],
        'final_test_loss': test_loss,
        'final_test_acc': test_acc,
        'train_time_min': train_time / 60,
        'history': result['history'],
    })
    
    print(f"  → Best (val): acc={result['best_val_acc']:.4f} at epoch {result['best_epoch']}")
    print(f"  → Final (test): acc={test_acc:.4f}, loss={test_loss:.4f}")
    print(f"  → Time: {train_time/60:.1f} min")

t_total = time.time() - t_start
print(f"\n{'='*60}")
print(f"Total training time: {t_total/60:.1f} minutes ({t_total/3600:.2f} hours)")

In [ ]:
# ============================================================
# Results Summary
# ============================================================

from scipy.stats import spearmanr

print("\n" + "="*70)
print("RESULTS SUMMARY")
print("="*70)
print(f"{'ID':<6} {'J_topo':<8} {'Params':<10} {'Best Val':<10} {'Final Test':<12} {'vs ResNet':<10}")
print("-"*70)

# Sort by J_topo
by_J = sorted(results, key=lambda x: -x['J_topo'])
for r in by_J:
    gap = r['final_test_acc'] - RESNET18_ACC
    comparable = '⭐⭐⭐' if r['final_test_acc'] >= RESNET18_ACC - 0.01 else (
                '⭐⭐' if r['final_test_acc'] >= RESNET18_ACC - 0.03 else (
                '⭐' if r['final_test_acc'] >= RESNET18_ACC - 0.05 else ''))
    print(f"{r['id']:<6} {r['J_topo']:<8.4f} {r['n_params']/1e6:<10.2f}M "
          f"{r['best_val_acc']:<10.4f} {r['final_test_acc']:<12.4f} {gap:>+9.4f} {comparable}")

print(f"\nReference: ResNet-18 → {RESNET18_ACC:.4f} (literature, 200 epochs)")
print("⭐⭐⭐ = within 1% of ResNet-18")
print("⭐⭐  = within 3% of ResNet-18")
print("⭐   = within 5% of ResNet-18")

# Sort by test accuracy
by_acc = sorted(results, key=lambda x: -x['final_test_acc'])
print("\nRanked by Test Accuracy (descending):")
for rank, r in enumerate(by_acc, 1):
    gap = r['final_test_acc'] - RESNET18_ACC
    print(f"  {rank}. {r['id']}: {r['final_test_acc']:.4f} (gap: {gap:+.4f})")

# Compute Spearman correlation: J_topo vs test accuracy
J_vals = [r['J_topo'] for r in results]
test_acc_vals = [r['final_test_acc'] for r in results]

r_spearman, p_val = spearmanr(J_vals, test_acc_vals)
print(f"\nSpearman(J_topo, test_acc): r = {r_spearman:.4f}, p = {p_val:.4f}")

print("\nINTERPRETATION:")
if abs(r_spearman) >= 0.7:
    print(f"  |r| >= 0.7: STRONG — J_topo predicts test accuracy")
elif abs(r_spearman) >= 0.5:
    print(f"  |r| >= 0.5: MODERATE — J_topo is useful but not definitive")
else:
    print(f"  |r| < 0.5: WEAK — J_topo alone insufficient")

In [ ]:
# ============================================================
# Save Results
# ============================================================

output = {
    'timestamp': datetime.now().isoformat(),
    'epochs': EPOCHS,
    'seed': SEED,
    'data_split': {
        'train': 45000,
        'val': 5000,
        'test': 10000
    },
    'reference_resnet18_acc': RESNET18_ACC,
    'candidates': [
        {k: v for k, v in r.items() if k != 'history'} 
        for r in results
    ],
    'spearman': {'r': r_spearman, 'p': p_val},
    'total_time_minutes': t_total / 60,
}

out_path = '/kaggle/working/phase_b_session1_results.json'
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f"Results saved to {out_path}")

In [ ]:
# ============================================================
# Visualization
# ============================================================

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Validation accuracy curves during training
ax1 = axes[0]
for r in results:
    ax1.plot(r['history']['val_acc'], label=f"{r['id']} (J={r['J_topo']:.3f})")
ax1.axhline(y=RESNET18_ACC, color='black', linestyle='--', alpha=0.5, label='ResNet-18 ref')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Validation Accuracy')
ax1.set_title('Validation Accuracy vs Epoch')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: J_topo vs Final Test Accuracy
ax2 = axes[1]
for r in results:
    ax2.scatter(r['J_topo'], r['final_test_acc'], s=120, c='blue')
    ax2.annotate(r['id'], (r['J_topo'], r['final_test_acc']), 
                 xytext=(8, 8), textcoords='offset points', fontsize=10)
ax2.axhline(y=RESNET18_ACC, color='black', linestyle='--', alpha=0.5, label='ResNet-18')
ax2.set_xlabel('J_topo (higher = better)')
ax2.set_ylabel('Test Accuracy (final)')
ax2.set_title(f'J_topo vs Test Accuracy\n(Spearman r={r_spearman:.3f}, p={p_val:.3f})')
ax2.grid(True, alpha=0.3)
ax2.legend()

# Plot 3: Params vs Test Accuracy (efficiency)
ax3 = axes[2]
for r in results:
    ax3.scatter(r['n_params']/1e6, r['final_test_acc'], s=120, c='green')
    ax3.annotate(r['id'], (r['n_params']/1e6, r['final_test_acc']),
                 xytext=(8, 8), textcoords='offset points', fontsize=10)
ax3.axhline(y=RESNET18_ACC, color='black', linestyle='--', alpha=0.5, label='ResNet-18')
ax3.set_xlabel('Parameters (M)')
ax3.set_ylabel('Test Accuracy (final)')
ax3.set_title('Parameter Efficiency')
ax3.grid(True, alpha=0.3)
ax3.legend()

plt.tight_layout()
plt.savefig('/kaggle/working/phase_b_session1.png', dpi=150, bbox_inches='tight')
plt.show()

print("Figure saved to /kaggle/working/phase_b_session1.png")